# Use Spark to predict credit risk with `ibm-watsonx-ai`

This notebook introduces commands for model persistence to watsonx.ai repository, model deployment, and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12 and Apache® Spark 3.5 with OpenJDK 17.

## Learning goals

The learning goals of this notebook are:

-  Load a CSV file into an Apache® Spark DataFrame.
-  Explore data.
-  Prepare data for training and evaluation.
-  Persist a pipeline and model in watsonx.ai repository from tar.gz files.
-  Deploy a model for online scoring using Wastson Machine Learning API.
-  Score sample scoring data using the watsonx.ai API.
-  Explore and visualize prediction result using the plotly package.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Load and explore data](#2.-Load-and-explore-data)
3. [Persist model](#3.-Persist-model)
4. [Predict locally](#4.-Predict-locally)
5. [Deploy model](#5.-Deploy-model)
6. [Score model](#6.-Score-model)
7. [Cleanup](#7.-Cleanup)
8. [Summary and next steps](#8.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install -U "pyspark==3.5.5" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

**Note:** If you're encountering `ModuleNotFoundError: No module named 'distutils'` exception, install the `setuptools` package and try again.

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click New Deployment Space
- Create an empty space
- Go to space `Settings` tab
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [5]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai, you need to set **space** which you will be using.

In [6]:
client.set.default_space(space_id)

'SUCCESS'

### Test Spark

In [7]:
try:
    from pyspark.sql import SparkSession
except:
    print(
        "Error: Spark runtime is missing. If you are using IBM watsonx, change the notebook runtime to Spark."
    )
    raise

<a id="2.-Load-and-explore-data"></a>
## 2. Load and explore data

In this section you will load the data as an Apache® Spark DataFrame and perform a basic exploration.
 

The csv file for German Credit Risk is available on the same repository as this notebook. Load the file to Apache® Spark DataFrame using code below.

In [8]:
import os

from wget import download

sample_dir = "spark_sample_model"
if not os.path.isdir(sample_dir):
    os.mkdir(sample_dir)

filename = os.path.join(sample_dir, "credit_risk_training.csv")
if not os.path.isfile(filename):
    filename = download(
        "https://github.com/IBM/watsonx-ai-samples/raw/master/cpd5.4/data/credit_risk/credit_risk_training.csv",
        out=sample_dir,
    )

In [9]:
spark = SparkSession.builder.getOrCreate()

df_data = (
    spark.read.format("org.apache.spark.sql.execution.datasources.csv.CSVFileFormat")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(filename)
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 11:05:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Explore the loaded data by using the following Apache® Spark DataFrame methods:
-  print schema
-  print top ten records
-  count all records

In [10]:
df_data.printSchema()

root
 |-- CheckingStatus: string (nullable = true)
 |-- LoanDuration: integer (nullable = true)
 |-- CreditHistory: string (nullable = true)
 |-- LoanPurpose: string (nullable = true)
 |-- LoanAmount: integer (nullable = true)
 |-- ExistingSavings: string (nullable = true)
 |-- EmploymentDuration: string (nullable = true)
 |-- InstallmentPercent: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- OthersOnLoan: string (nullable = true)
 |-- CurrentResidenceDuration: integer (nullable = true)
 |-- OwnsProperty: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- InstallmentPlans: string (nullable = true)
 |-- Housing: string (nullable = true)
 |-- ExistingCreditsCount: integer (nullable = true)
 |-- Job: string (nullable = true)
 |-- Dependents: integer (nullable = true)
 |-- Telephone: string (nullable = true)
 |-- ForeignWorker: string (nullable = true)
 |-- Risk: string (nullable = true)



As you can see, the data contains 21 fields. Risk field is the one we would like to predict (label).

In [11]:
df_data.show(n=5, truncate=False, vertical=True)

-RECORD 0------------------------------------------
 CheckingStatus           | 0_to_200               
 LoanDuration             | 31                     
 CreditHistory            | credits_paid_to_date   
 LoanPurpose              | other                  
 LoanAmount               | 1889                   
 ExistingSavings          | 100_to_500             
 EmploymentDuration       | less_1                 
 InstallmentPercent       | 3                      
 Sex                      | female                 
 OthersOnLoan             | none                   
 CurrentResidenceDuration | 3                      
 OwnsProperty             | savings_insurance      
 Age                      | 32                     
 InstallmentPlans         | none                   
 Housing                  | own                    
 ExistingCreditsCount     | 1                      
 Job                      | skilled                
 Dependents               | 1                      
 Telephone  

In [12]:
print("Number of records: " + str(df_data.count()))

Number of records: 5000


As you can see, the data set contains 5000 records.

### 2.1 Prepare data

In this subsection you will split your data into: train, test and predict datasets.

In [13]:
splitted_data = df_data.randomSplit([0.8, 0.18, 0.02], 24)
train_data = splitted_data[0]
test_data = splitted_data[1]
predict_data = splitted_data[2]

print("Number of training records: " + str(train_data.count()))
print("Number of testing records : " + str(test_data.count()))
print("Number of prediction records : " + str(predict_data.count()))

Number of training records: 4005
Number of testing records : 901
Number of prediction records : 94


As you can see our data has been successfully split into three datasets: 

-  The train data set, which is the largest group, is used for training.
-  The test data set will be used for model evaluation and is used to test the assumptions of the model.
-  The predict data set will be used for prediction.

<a id="3.-Persist-model"></a>
## 3. Persist model

In this section you will learn how to store your pipeline and model in watsonx.ai repository by using Python client libraries.

**Note**: Apache® Spark 3.5 is required.

### 3.1: Save pipeline and model

In this subsection you will learn how to save pipeline and model artifacts to your watsonx.ai instance.

**Download pipeline and model archives**

In [14]:
import os

from wget import download

sample_dir = "spark_sample_model"
if not os.path.isdir(sample_dir):
    os.mkdir(sample_dir)

pipeline_filename = os.path.join(sample_dir, "credit_risk_spark_pipeline.tar.gz")
if not os.path.isfile(pipeline_filename):
    pipeline_filename = download(
        "https://github.com/IBM/watsonx-ai-samples/raw/master/cpd5.4/models/spark/credit-risk/model/credit_risk_spark_pipeline.tar.gz",
        out=sample_dir,
    )
model_filename = os.path.join(sample_dir, "credit_risk_spark_model.gz")
if not os.path.isfile(model_filename):
    model_filename = download(
        "https://github.com/IBM/watsonx-ai-samples/raw/master/cpd5.4/models/spark/credit-risk/model/credit_risk_spark_model.gz",
        out=sample_dir,
    )

**Store piepline and model**

To be able to store your Spark model, you need to provide a training data reference, this will allow to read the model schema automatically.

In [15]:
training_data_references = [
    {
        "type": "fs",
        "connection": {},
        "location": {},
        "schema": {
            "id": "training_schema",
            "fields": [
                {
                    "metadata": {},
                    "name": "CheckingStatus",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "LoanDuration",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "CreditHistory",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "LoanPurpose",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "LoanAmount",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "ExistingSavings",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "EmploymentDuration",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "InstallmentPercent",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "Sex",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "OthersOnLoan",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "CurrentResidenceDuration",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "OwnsProperty",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "Age",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "InstallmentPlans",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "Housing",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "ExistingCreditsCount",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "Job",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "Dependents",
                    "nullable": True,
                    "type": "integer",
                },
                {
                    "metadata": {},
                    "name": "Telephone",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {},
                    "name": "ForeignWorker",
                    "nullable": True,
                    "type": "string",
                },
                {
                    "metadata": {"modeling_role": "target"},
                    "name": "Risk",
                    "nullable": True,
                    "type": "string",
                },
            ],
        },
    }
]

In [16]:
published_model_details = client.repository.store_model(
    model=model_filename,
    meta_props={
        client.repository.ModelMetaNames.NAME: "Credit Risk model",
        client.repository.ModelMetaNames.TYPE: "mllib_3.5",
        client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: client.software_specifications.get_id_by_name(
            "spark-mllib_3.5"
        ),
        client.repository.ModelMetaNames.TRAINING_DATA_REFERENCES: training_data_references,
        client.repository.ModelMetaNames.LABEL_FIELD: "Risk",
    },
    training_data=train_data,
    pipeline=pipeline_filename,
)

In [17]:
model_id = client.repository.get_model_id(published_model_details)

model_id

'ee2629ee-0f31-440a-8fce-c172d0868cd4'

In [18]:
client.repository.get_model_details(model_id)

{'metadata': {'name': 'Credit Risk model',
  'space_id': '892f33cd-5522-4944-83c5-9505b801df9b',
  'resource_key': '1c4f655f-12ab-4b5a-8e7e-d621e0a5b14c',
  'id': 'ee2629ee-0f31-440a-8fce-c172d0868cd4',
  'created_at': '2026-04-21T11:05:56Z',
  'rov': {'member_roles': {'1000331001': {'user_iam_id': '1000331001',
     'roles': ['OWNER']}}},
  'owner': '1000331001'},
 'entity': {'software_spec': {'id': 'e8cd7001-fd05-5eea-b697-0a3bff9cf51f'},
  'type': 'mllib_3.5',
  'training_data_references': [{'type': 'fs',
    'connection': {},
    'location': {},
    'schema': {'id': 'training_schema',
     'fields': [{'name': 'CheckingStatus',
       'type': 'string',
       'nullable': True,
       'metadata': {}},
      {'name': 'LoanDuration',
       'type': 'integer',
       'nullable': True,
       'metadata': {}},
      {'name': 'CreditHistory',
       'type': 'string',
       'nullable': True,
       'metadata': {}},
      {'name': 'LoanPurpose',
       'type': 'string',
       'nullable': T

Get saved model metadata from watsonx.ai.

**Tip**: Use `client.repository.ModelMetaNames.show()` to get the list of available props.

In [19]:
client.repository.ModelMetaNames.show()

------------------------  ----  --------  ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
META_PROP NAME            TYPE  REQUIRED  SCHEMA
NAME                      str   Y
DESCRIPTION               str   N
INPUT_DATA_SCHEMA         list  N         {'id(required)': 'string', 'fields(required)': [{'name(required)': 'string', 'type(required)': 'string', 'nullable(optional)': 'string'}]}
TRAINING_DATA_REFERENCES  list  N         [{'name(optional)': 'string', 'type(required)': 'string', 'connection(required)': {'endpoint_url(required)': 'string', 'access_key_id(required)': 'string', 'secret_access_key(required)': 'string'},

### 3.2: Load model

In this subsection you will learn how to load back saved model from specified instance of watsonx.ai.

In [20]:
from pyspark.ml.pipeline import PipelineModel

loaded_model: PipelineModel = client.repository.load(model_id)

Verify that the model was loaded correctly.

In [21]:
assert isinstance(loaded_model, PipelineModel)

<a id="4.-Predict-locally"></a>
## 4. Predict locally

In this section you will learn how to score test data using loaded model.

### 4.1: Make local prediction using previously loaded model and test data

In this subsection you will score *predict_data* data set.

In [22]:
predictions = loaded_model.transform(predict_data)

26/04/21 11:06:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Preview the results by calling the *show()* method on the predictions DataFrame.

In [23]:
predictions.show(5, vertical=True)

-RECORD 0----------------------------------------
 CheckingStatus           | 0_to_200             
 LoanDuration             | 4                    
 CreditHistory            | all_credits_paid_... 
 LoanPurpose              | education            
 LoanAmount               | 936                  
 ExistingSavings          | less_100             
 EmploymentDuration       | less_1               
 InstallmentPercent       | 2                    
 Sex                      | male                 
 OthersOnLoan             | none                 
 CurrentResidenceDuration | 2                    
 OwnsProperty             | savings_insurance    
 Age                      | 41                   
 InstallmentPlans         | bank                 
 Housing                  | rent                 
 ExistingCreditsCount     | 1                    
 Job                      | unskilled            
 Dependents               | 1                    
 Telephone                | none                 


By tabulating a count, you can see which product line is the most popular.

In [24]:
predictions.select("predictedLabel").groupBy("predictedLabel").count().show(
    truncate=False
)

+--------------+-----+
|predictedLabel|count|
+--------------+-----+
|No Risk       |71   |
|Risk          |23   |
+--------------+-----+



<a id="5.-Deploy-model"></a>
## 5. Deploy model

In this step, we will create both an online deployment and a batch deployment of Apache® Spark model using `ibm-watsonx-ai`. Depending on your use-case, only one deployment out of these two might be necessary. You can learn more about batch deployments [here](https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/deploy-batch-details.html?context=cpdaas).

**Note:** You can also use REST API to deploy and score. For more information about REST APIs, see the [Swagger Documentation](https://cloud.ibm.com/apidocs/machine-learning#deployments-create).

### 5.1: Create online scoring endpoint

Create online deployment for published model

In [25]:
online_deployment = client.deployments.create(
    model_id,
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: "Credit Risk model online deployment",
        client.deployments.ConfigurationMetaNames.ONLINE: {},
    },
)



######################################################################################

Synchronous deployment creation for id: 'ee2629ee-0f31-440a-8fce-c172d0868cd4' started

######################################################################################


initializing
Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
.......
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='2967a406-a402-4293-89be-ebb0127985da'
-----------------------------------------------------------------------------------------------




You can retrieve now your online deployment ID

In [26]:
online_deployment_id = client.deployments.get_id(online_deployment)

You can also list all deployments in your space

In [27]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,2967a406-a402-4293-89be-ebb0127985da,Credit Risk model online deployment,ready,2026-04-21T11:06:27.154Z,model,supported,


If you want to get additional information on your deployment, you can do it as below

In [28]:
client.deployments.get_details(online_deployment_id)

### 5.2 Create batch scoring endpoint

Create batch deployment for published model

In [29]:
batch_deployment = client.deployments.create(
    artifact_uid=model_id,
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: "Credit Risk model batch deployment",
        client.deployments.ConfigurationMetaNames.BATCH: {},
        client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {
            "name": "S",
            "num_nodes": 1,
        },
    },
)



######################################################################################

Synchronous deployment creation for id: 'ee2629ee-0f31-440a-8fce-c172d0868cd4' started

######################################################################################


ready
Note: The `/text/generation` and `/text/generation_stream` endpoints will be deprecated in an upcoming release and removed in a future release.
.


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='5fa11cae-8276-4d75-a3a5-c996cb27e3d8'
-----------------------------------------------------------------------------------------------




You can retrieve now your batch deployment ID

In [30]:
batch_deployment_id = client.deployments.get_id(batch_deployment)

You can also list all deployments in your space

In [31]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,5fa11cae-8276-4d75-a3a5-c996cb27e3d8,Credit Risk model batch deployment,ready,2026-04-21T11:07:09.012Z,model,supported,
1,2967a406-a402-4293-89be-ebb0127985da,Credit Risk model online deployment,ready,2026-04-21T11:06:27.154Z,model,supported,


If you want to get additional information on your deployment, you can do it as below

In [32]:
client.deployments.get_details(batch_deployment_id)

Note: The `/text/generation` and `/text/generation_stream` endpoints will be deprecated in an upcoming release and removed in a future release.


{'entity': {'asset': {'id': 'ee2629ee-0f31-440a-8fce-c172d0868cd4'},
  'batch': {},
  'chat_enabled': False,
  'custom': {},
  'deployed_asset_type': 'model',
  'hardware_spec': {'name': 'S', 'num_nodes': 1},
  'name': 'Credit Risk model batch deployment',
  'space_id': '892f33cd-5522-4944-83c5-9505b801df9b',
  'status': {'state': 'ready'}},
 'metadata': {'created_at': '2026-04-21T11:07:09.012Z',
  'id': '5fa11cae-8276-4d75-a3a5-c996cb27e3d8',
  'modified_at': '2026-04-21T11:07:09.012Z',
  'name': 'Credit Risk model batch deployment',
  'owner': '1000331001',
  'space_id': '892f33cd-5522-4944-83c5-9505b801df9b'},
 'system': {'warnings': [{'id': 'text_generation_deprecation',
    'message': 'The `/text/generation` and `/text/generation_stream` endpoints will be deprecated in an upcoming release and removed in a future release.'}]}}

<a id="6.-Score-model"></a>
## 6. Score model

Prepare scoring payload

In [33]:
import json

fields = [
    "CheckingStatus",
    "LoanDuration",
    "CreditHistory",
    "LoanPurpose",
    "LoanAmount",
    "ExistingSavings",
    "EmploymentDuration",
    "InstallmentPercent",
    "Sex",
    "OthersOnLoan",
    "CurrentResidenceDuration",
    "OwnsProperty",
    "Age",
    "InstallmentPlans",
    "Housing",
    "ExistingCreditsCount",
    "Job",
    "Dependents",
    "Telephone",
    "ForeignWorker",
]

values = [
    [
        "no_checking",
        13,
        "credits_paid_to_date",
        "car_new",
        1343,
        "100_to_500",
        "1_to_4",
        2,
        "female",
        "none",
        3,
        "savings_insurance",
        46,
        "none",
        "own",
        2,
        "skilled",
        1,
        "none",
        "yes",
    ],
    [
        "no_checking",
        24,
        "prior_payments_delayed",
        "furniture",
        4567,
        "500_to_1000",
        "1_to_4",
        4,
        "male",
        "none",
        4,
        "savings_insurance",
        36,
        "none",
        "free",
        2,
        "management_self-employed",
        1,
        "none",
        "yes",
    ],
    [
        "0_to_200",
        26,
        "all_credits_paid_back",
        "car_new",
        863,
        "less_100",
        "less_1",
        2,
        "female",
        "co-applicant",
        2,
        "real_estate",
        38,
        "none",
        "own",
        1,
        "skilled",
        1,
        "none",
        "yes",
    ],
    [
        "0_to_200",
        14,
        "no_credits",
        "car_new",
        2368,
        "less_100",
        "1_to_4",
        3,
        "female",
        "none",
        3,
        "real_estate",
        29,
        "none",
        "own",
        1,
        "skilled",
        1,
        "none",
        "yes",
    ],
    [
        "0_to_200",
        4,
        "no_credits",
        "car_new",
        250,
        "less_100",
        "unemployed",
        2,
        "female",
        "none",
        3,
        "real_estate",
        23,
        "none",
        "rent",
        1,
        "management_self-employed",
        1,
        "none",
        "yes",
    ],
    [
        "no_checking",
        17,
        "credits_paid_to_date",
        "car_new",
        832,
        "100_to_500",
        "1_to_4",
        2,
        "male",
        "none",
        2,
        "real_estate",
        42,
        "none",
        "own",
        1,
        "skilled",
        1,
        "none",
        "yes",
    ],
    [
        "no_checking",
        33,
        "outstanding_credit",
        "appliances",
        5696,
        "unknown",
        "greater_7",
        4,
        "male",
        "co-applicant",
        4,
        "unknown",
        54,
        "none",
        "free",
        2,
        "skilled",
        1,
        "yes",
        "yes",
    ],
    [
        "0_to_200",
        13,
        "prior_payments_delayed",
        "retraining",
        1375,
        "100_to_500",
        "4_to_7",
        3,
        "male",
        "none",
        3,
        "real_estate",
        37,
        "none",
        "own",
        2,
        "management_self-employed",
        1,
        "none",
        "yes",
    ],
]

meta_props = {
    client.deployments.ScoringMetaNames.INPUT_DATA: [
        {
            "fields": fields,
            "values": values,
        }
    ]
}

### Online deployment scoring
Scoring of online deployments can be performed using the `score` method.

In [34]:
predictions = client.deployments.score(online_deployment_id, meta_props)

print(json.dumps(predictions, indent=2))

{
  "predictions": [
    {
      "fields": [
        "CheckingStatus",
        "LoanDuration",
        "CreditHistory",
        "LoanPurpose",
        "LoanAmount",
        "ExistingSavings",
        "EmploymentDuration",
        "InstallmentPercent",
        "Sex",
        "OthersOnLoan",
        "CurrentResidenceDuration",
        "OwnsProperty",
        "Age",
        "InstallmentPlans",
        "Housing",
        "ExistingCreditsCount",
        "Job",
        "Dependents",
        "Telephone",
        "ForeignWorker",
        "CheckingStatus_IX",
        "CreditHistory_IX",
        "LoanPurpose_IX",
        "ExistingSavings_IX",
        "EmploymentDuration_IX",
        "Sex_IX",
        "OthersOnLoan_IX",
        "OwnsProperty_IX",
        "InstallmentPlans_IX",
        "Housing_IX",
        "Job_IX",
        "Telephone_IX",
        "ForeignWorker_IX",
        "features",
        "rawPrediction",
        "probability",
        "prediction",
        "predictedLabel"
      ],
      "

### Batch deployment scoring

In order to score a model in batch deployment, a job needs to be created.

In [35]:
job = client.deployments.create_job(batch_deployment_id, meta_props=meta_props)

Note: The `/text/generation` and `/text/generation_stream` endpoints will be deprecated in an upcoming release and removed in a future release.


After submitting your job, you can retrieve its ID

In [36]:
job_id = client.deployments.get_job_id(job)

You can also list all jobs in your space.

In [37]:
client.deployments.list_jobs()

,JOB-ID,STATE,CREATED,DEPLOYMENT-ID
0,bf053aa2-82f4-498d-8101-5b81d26d458b,queued,2026-04-21T11:07:45.320Z,5fa11cae-8276-4d75-a3a5-c996cb27e3d8


If you want to get additional information on your job, you can do it as below.

In [38]:
client.deployments.get_job_details(job_id)

{'entity': {'deployment': {'id': '5fa11cae-8276-4d75-a3a5-c996cb27e3d8'},
  'platform_job': {'job_id': '5507b563-455b-4133-b894-a35fa379bf20',
   'run_id': 'bf053aa2-82f4-498d-8101-5b81d26d458b'},
  'scoring': {'input_data': [{'fields': ['CheckingStatus',
      'LoanDuration',
      'CreditHistory',
      'LoanPurpose',
      'LoanAmount',
      'ExistingSavings',
      'EmploymentDuration',
      'InstallmentPercent',
      'Sex',
      'OthersOnLoan',
      'CurrentResidenceDuration',
      'OwnsProperty',
      'Age',
      'InstallmentPlans',
      'Housing',
      'ExistingCreditsCount',
      'Job',
      'Dependents',
      'Telephone',
      'ForeignWorker'],
     'values': [['no_checking',
       13,
       'credits_paid_to_date',
       'car_new',
       1343,
       '100_to_500',
       '1_to_4',
       2,
       'female',
       'none',
       3,
       'savings_insurance',
       46,
       'none',
       'own',
       2,
       'skilled',
       1,
       'none',
       '

Here you can check status of your batch scoring.

In [39]:
import time

elapsed_time = 0
while (
    client.deployments.get_job_status(job_id).get("state") != "completed"
    and elapsed_time < 300
):
    print(f" Current state: {client.deployments.get_job_status(job_id).get('state')}")
    elapsed_time += 10
    time.sleep(10)

if client.deployments.get_job_status(job_id).get("state") == "completed":
    print(f" Current state: {client.deployments.get_job_status(job_id).get('state')}")
    job_details_do = client.deployments.get_job_details(job_id)
    print(job_details_do)
else:
    print("Job hasn't completed successfully in 5 minutes.")

 Current state: queued
 Current state: queued
 Current state: queued
 Current state: queued
 Current state: running
 Current state: completed
{'entity': {'deployment': {'id': '5fa11cae-8276-4d75-a3a5-c996cb27e3d8'}, 'platform_job': {'job_id': '5507b563-455b-4133-b894-a35fa379bf20', 'run_id': 'bf053aa2-82f4-498d-8101-5b81d26d458b'}, 'scoring': {'input_data': [{'fields': ['CheckingStatus', 'LoanDuration', 'CreditHistory', 'LoanPurpose', 'LoanAmount', 'ExistingSavings', 'EmploymentDuration', 'InstallmentPercent', 'Sex', 'OthersOnLoan', 'CurrentResidenceDuration', 'OwnsProperty', 'Age', 'InstallmentPlans', 'Housing', 'ExistingCreditsCount', 'Job', 'Dependents', 'Telephone', 'ForeignWorker'], 'values': [['no_checking', 13, 'credits_paid_to_date', 'car_new', 1343, '100_to_500', '1_to_4', 2, 'female', 'none', 3, 'savings_insurance', 46, 'none', 'own', 2, 'skilled', 1, 'none', 'yes'], ['no_checking', 24, 'prior_payments_delayed', 'furniture', 4567, '500_to_1000', '1_to_4', 4, 'male', 'none', 4

After the job completes, you can retrieve its scoring data

In [40]:
job_details = client.deployments.get_job_details(job_id)

print(json.dumps(job_details, indent=2))

{
  "entity": {
    "deployment": {
      "id": "5fa11cae-8276-4d75-a3a5-c996cb27e3d8"
    },
    "platform_job": {
      "job_id": "5507b563-455b-4133-b894-a35fa379bf20",
      "run_id": "bf053aa2-82f4-498d-8101-5b81d26d458b"
    },
    "scoring": {
      "input_data": [
        {
          "fields": [
            "CheckingStatus",
            "LoanDuration",
            "CreditHistory",
            "LoanPurpose",
            "LoanAmount",
            "ExistingSavings",
            "EmploymentDuration",
            "InstallmentPercent",
            "Sex",
            "OthersOnLoan",
            "CurrentResidenceDuration",
            "OwnsProperty",
            "Age",
            "InstallmentPlans",
            "Housing",
            "ExistingCreditsCount",
            "Job",
            "Dependents",
            "Telephone",
            "ForeignWorker"
          ],
          "values": [
            [
              "no_checking",
              13,
              "credits_paid_to_date",

<a id="7.-Cleanup"></a>
## 7. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="8.-Summary-and-next-steps"></a>
## 8. Summary and next steps

You successfully completed this notebook! You learned how to use Apache Spark machine learning as well as Watson Machine Learning for model creation and deployment. 
 
Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Amadeusz Masny**, Python Software Developer at watsonx.ai

**Rafał Chrzanowski**, Software Engineer Intern at watsonx.ai.

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.